# **Modelo LightGCN + Texto**
### Proyecto Hito 3
### Sistemas Recomendadores IIC3633-1 2025-2
### **Grupo 3:** 

- Nicolás Antonio Bueno Abett de la Torre 

- Felipe Andrés Fuentes González

- Jorge Andrés Jacque Palma

- Francisco Nicolás Solís Gormaz

## Índice

>[0- Instalación de librerías](#0--instalación-de-librerías)

>[1- Carga de datos](#1--carga-de-datos)

>[2- Multimodalidad con texto: embeddings de descripciones de videojuegos](#2--multimodalidad-con-texto-embeddings-de-descripciones-de-videojuegos)

>[3- Definición del modelo LightGCN, formateo de datos, y entrenamiento](#3--definición-del-modelo-lightgcn-formateo-de-datos-y-entrenamiento)

>[4- Obtención de scores de LightGCN](#4--obtención-de-scores-de-lightgcn)

>[5- Cálculo de scores combinados (multimodales)](#5--cálculo-de-scores-combinados-multimodales)

>[6- Generación de nuevas recomendaciones con scores combinados (multimodales)](#6--generación-de-nuevas-recomendaciones-con-scores-combinados-multimodales)

>[7- Métricas](#7--métricas)

>[8- Referencias](#8--referencias)

## 0- Instalación de librerías

En caso de usar Colab, correr estas celdas. Si se corre en local, se deben tener exactamente las mismas versiones de las librerías indicadas en estas celdas. Las versiones de las librerías son:

- numpy: 1.25.0
- pandas: 2.2.2
- scipy: 1.10.1
- tqdm: 4.66.6
- torch: 2.5.1+cpu
- tensorboard: 2.12.3
- recbole: 1.2.1
- transformers: 4.41.2
- sentence-transformers: 2.6.1

In [ ]:
# !pip uninstall -y numpy
# !pip install numpy==1.25

In [ ]:
# !pip uninstall -y pandas
# !pip install pandas==2.2.2

In [ ]:
# !pip uninstall -y scipy
# !pip install scipy==1.10.1

In [ ]:
# !pip uninstall -y tqdm
# !pip install tqdm==4.66.6

In [ ]:
# !pip uninstall -y torch
# !pip uninstall -y torchvision
# !pip uninstall -y torchaudio
# !pip install torch==2.5.1+cpu --index-url https://download.pytorch.org/whl/cpu

In [ ]:
# #Necesario para RecBole, también instala tensorboard-data-server 0.7.2
# !pip uninstall -y tensorboard
# !pip install tensorboard==2.12.3

In [ ]:
# !pip uninstall -y recbole
# !pip install recbole==1.2.1

In [ ]:
# #(para sentence-tranformer)
# !pip uninstall -y transformers
# !pip install transformers==4.41.2

In [ ]:
# #(sin usar tensorflow - keras)
# !pip uninstall -y sentence-transformers
# !pip install sentence-transformers==2.6.1

## 1- Carga de datos

Se leen los archivos de datos (entrenamiento, testeo, y validación) correspondientes al muestreo del dataset principal del proyecto ("Game Recommendations on Steam: A dataset of games, users and reviews for building recommendation systems". Anton Kozyriev, 2024. Kaggle.) [7] y se almacenan en un dataframe:

In [1]:
import pandas as pd

df_train = pd.read_csv('train_split.csv')
df_test = pd.read_csv('test_split.csv')
df_val = pd.read_csv('val_split.csv')

La estructura de estos archivos es:

In [2]:
df_train.head(5)

,app_id,helpful,funny,date,is_recommended,hours,user_id,review_id
0,322330,0,0,2019-07-02,True,67.5,731,33484606
1,433340,0,0,2020-01-24,True,32.3,731,26236770
2,394360,2,0,2020-04-20,True,403.7,731,25992499
3,4700,0,0,2020-04-21,True,683.5,731,9845461
4,246090,3,0,2015-02-25,False,7.9,3128,28148695


Dataframe de games.csv para mostrar nombres de juegos recomendados

In [ ]:
df_games = pd.read_csv('games.csv')

En este dataframe se guarda la información del archivo json de metadata de videojuegos:

In [2]:
games_metadata = pd.read_json('games_metadata.json', lines=True)
games_metadata.head(5)

,app_id,description,tags
0,13500,Enter the dark underworld of Prince of Persia ...,"[Action, Adventure, Parkour, Third Person, Gre..."
1,22364,,[Action]
2,113020,Monaco: What's Yours Is Mine is a single playe...,"[Co-op, Stealth, Indie, Heist, Local Co-Op, St..."
3,226560,Escape Dead Island is a Survival-Mystery adven...,"[Zombies, Adventure, Survival, Action, Third P..."
4,249050,Dungeon of the Endless is a Rogue-Like Dungeon...,"[Roguelike, Strategy, Tower Defense, Pixel Gra..."


Dado que el único feedback explícito que se posee en los datasets de interacciones es la columna "is_recommended", se añade una columna "rating" cuyo valor es 1 si "is_recommended" es "True", y su valor es 0 si "is_recommended" es "False". Esto se hace para cada uno de los df:

In [3]:
regla_rating = {True: 1, False: 0}

df_train['rating'] = df_train['is_recommended'].map(regla_rating)
df_test['rating'] = df_test['is_recommended'].map(regla_rating)
df_val['rating'] = df_val['is_recommended'].map(regla_rating)

El resultado es:

In [5]:
df_train.head(5)

,app_id,helpful,funny,date,is_recommended,hours,user_id,review_id,rating
0,322330,0,0,2019-07-02,True,67.5,731,33484606,1
1,433340,0,0,2020-01-24,True,32.3,731,26236770,1
2,394360,2,0,2020-04-20,True,403.7,731,25992499,1
3,4700,0,0,2020-04-21,True,683.5,731,9845461,1
4,246090,3,0,2015-02-25,False,7.9,3128,28148695,0


## 2- Multimodalidad con texto: embeddings de descripciones de videojuegos

En este sección se generarán embeddings de las descripciones de los videojuegos mediante Sentence Transformers. Estos embeddings de juegos se compararán con los embeddings promedios/normalizados de los usuarios (usando los embeddings de las descripciones de juegos relevantes para ellos) y luego calculando su similitud. Esta similitud se usará para obtener un score por cada par usuario-ítem, y luego incluir este score junto al de LightGCN para generar recomendaciones.

In [4]:
from sentence_transformers import SentenceTransformer

model_transformer = SentenceTransformer('all-mpnet-base-v2')

c:\Users\felip\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Ahora, se guardan las descripciones de cada videojuego, se obtienen sus embeddings mediante el Sentence Tranformer, y luego se guardan en un diccionario para acceder a ellas a partir de su id. Para hacer más eficiente este proceso, se toman solo las descripciones de los videojuegos incluídos en el muestreo, es decir, las de todos los videojuegos presentes en los dataframes de entrenamiento, testeo, y validación. El dicionario se guarda en un archivo Pickle para su uso posterior.

Opción 1: calcular todos los embeddings y luego filtrar (alrededor de 30 min).

No es necesario volver a ejecutar el código de abajo, basta con cargar el archivo Pickle del diccionario de los embeddings en la celda de código de más abajo.

In [ ]:
# #opción para calcular todos los embeddings de los videojuegos

# game_descriptions = games_metadata["description"].tolist()

# game_embeddings = model_transformer.encode(game_descriptions, show_progress_bar=True, convert_to_numpy=True)

# game_embedding_dict = {int(item_id):emb for item_id, emb in zip(games_metadata["app_id"], game_embeddings)}

# import pickle

# # #para guardar el diccionario completo de embeddings (de todos los videojuegos)
# # with open("game_embedding_dict.pkl", "wb") as f:
# #     pickle.dump(game_embedding_dict, f)

# #para guardar el diccionario de embeddings incluyendo solo los videojuegos que se usan en el modelo (en los dataframes de train, test, val)
# id_utiles = set(df_train['app_id']).union(set(df_test['app_id'])).union(set(df_val['app_id']))
# game_embedding_dict_util = {k:v for k,v in game_embedding_dict.items() if k in id_utiles}
# with open("game_embedding_dict_util.pkl", "wb") as f:
#     pickle.dump(game_embedding_dict_util, f)

Batches:   0%|          | 0/1590 [00:00<?, ?it/s]

Opción 2: calcular solo los embeddings de los videojuegos del muestreo en los dataframes de train-test-validation (alrededor de 5 min).

No es necesario volver a ejecutar el código de abajo, basta con cargar el archivo Pickle del diccionario de los embeddings en la celda de código de más abajo.

In [ ]:
# #opción para calcular sólo los embeddings de los videojuegos que realmente se usan en el modelo (en los dataframes de train, test, val)
# id_utiles = set(df_train['app_id']).union(set(df_test['app_id'])).union(set(df_val['app_id']))

# games_metadata_filtrado = games_metadata[games_metadata["app_id"].isin(id_utiles)]
# game_descriptions = games_metadata_filtrado["description"].tolist()

# game_embeddings = model_transformer.encode(game_descriptions, show_progress_bar=True, convert_to_numpy=True)

# game_embedding_dict_util = {int(item_id):emb for item_id, emb in zip(games_metadata["app_id"], game_embeddings)}

# import pickle

# with open("game_embedding_dict_util.pkl", "wb") as f:
#     pickle.dump(game_embedding_dict_util, f)


Si ya se posee el archivo Pickle del diccionario de los embeddings, se debe ejecutar la siguiente celda de código:

In [5]:
import pickle

# #para cargar el diccionario completo de embeddings (de todos los videojuegos)
# with open("game_embedding_dict.pkl", "rb") as f:
#     game_embedding_dict = pickle.load(f)

#para cargar el diccionario de embeddings incluyendo solo los videojuegos que se usan en el modelo (en los dataframes de train, test, val)
with open("game_embedding_dict_util.pkl", "rb") as f:
    game_embedding_dict = pickle.load(f)

La cantidad de videojuegos para los que se usará su embedding es:

In [6]:
len(game_embedding_dict)

2880

Ahora, se deben calcular los embeddings de los usuarios (o embeddings de perfil de usuario). La gracia de esto es tomar para cada usuario los embeddings de los videojuegos con los que interactuó en el dataset de entrenamiento (df_train) y que evaluó positivamente (rating = 1, es decir, el usuario en su review "recomienda el juego"). Luego, una vez se tienen todos los embeddings de descripciones de videojuegos "positivos" para cierto usuario, se calcula el promedio y se almacena en el diccionario "user_profile_embeddings" para acceder a ese embedding "promedio" a partir de la id de usuario.

In [7]:
import numpy as np

embedding_dim = int(model_transformer.get_sentence_embedding_dimension())

user_profile_embeddings = {}

for user_id, grupo in df_train.groupby("user_id"):
    #solo interacciones positivas
    grupo_positivo = grupo[grupo["rating"] == 1]

    item_ids_positivos = grupo_positivo["app_id"].tolist()

    embeddings_positivos = [game_embedding_dict[item] for item in item_ids_positivos if item in game_embedding_dict]

    if len(embeddings_positivos) == 0:
        #usuario sin interacciones positivas en train
        user_profile_embeddings[user_id] = np.zeros(embedding_dim)
    else:
        user_profile_embeddings[user_id] = np.mean(embeddings_positivos, axis=0)

Usuarios con embedding promedio calculado:

In [8]:
len(user_profile_embeddings)

9906

Ahora, se deben calcular las similitudes entre el embedding promedio de cada usuario y el embedding de cada videojuego del dataset. Cada una de estas similitudes calculadas será el "score" de cada par usuario-ítem, que más adelante se combinará con los scores de LightGCN para obtener un score combinado (multimodal) con el que finalmente se generarán las listas de recomendación top 10 (rankings). Se considerará que una similitud más alta entre un embedding de usuario y uno de un ítem indica que a ese usuario se le debería recomendar ese ítem. Los scores (similitudes) se guardan para cada usuario en el diccionario "scores_user_item_embeddings". Este diccionario tiene como llave la id de usuario y como valor otro diccionario donde se tiene la id del ítem y su correspondiente score de similitud para el usuario determinado.

In [9]:
import numpy as np

def compute_user_item_scores(user_profile_embeddings, game_embedding_dict):
    #retorna un dict: user_id -> dict(item_id -> score_coseno)

    #game embeddings a una matriz ordenada
    item_ids = list(game_embedding_dict.keys())
    item_emb_matrix = np.vstack([game_embedding_dict[i] for i in item_ids])

    #normalizar ítems para acelerar similitud
    item_emb_norm = item_emb_matrix / np.linalg.norm(item_emb_matrix, axis=1, keepdims=True)

    #diccionario de scores final
    dict_scores = {}

    for user_id, user_emb in user_profile_embeddings.items():

        #normalizar embedding de usuario
        norm = np.linalg.norm(user_emb)
        if norm == 0:
            #sin normalizar si su norma es 0 (vector nulo):
            user_emb_norm = user_emb
        else:
            user_emb_norm = user_emb / norm 

        #calcular similitud coseno con todos los ítems
        scores = np.dot(item_emb_norm, user_emb_norm)

        #guardar como dict: item_id → score
        user_scores = {item_id: float(score) for item_id, score in zip(item_ids, scores)}

        dict_scores[user_id] = user_scores

    return dict_scores

scores_user_item_embeddings = compute_user_item_scores(user_profile_embeddings, game_embedding_dict)

## 3- Definición del modelo LightGCN, formateo de datos, y entrenamiento

Se formatean los datos para la librería RecBole:

In [ ]:
import os

#carpeta del dataset
dataset_name = "dataset_recbole"
dataset_dir = f"./dataset_recbole"
os.makedirs(dataset_dir, exist_ok=True)

#se formatean los dataframes para que tengan las columnas requeridas por RecBole
df_train_lightgcn = df_train[["user_id", "app_id", "rating"]].copy()
df_test_lightgcn = df_test[["user_id", "app_id", "rating"]].copy()
df_val_lightgcn = df_val[["user_id", "app_id", "rating"]].copy()
df_train_lightgcn.rename(columns={
    "user_id": "user_id:token",
    "app_id": "item_id:token",
    "rating": "rating:float"}, inplace=True)
df_test_lightgcn.rename(columns={
    "user_id": "user_id:token",
    "app_id": "item_id:token",
    "rating": "rating:float"}, inplace=True)
df_val_lightgcn.rename(columns={
    "user_id": "user_id:token",
    "app_id": "item_id:token",
    "rating": "rating:float"}, inplace=True)

#se guarda cada split como archivo .inter en la carpeta definida
df_train_lightgcn.to_csv(f"{dataset_dir}/{dataset_name}.train.inter", sep="\t", index=False)
df_test_lightgcn.to_csv(f"{dataset_dir}/{dataset_name}.test.inter", sep="\t", index=False)
df_val_lightgcn.to_csv(f"{dataset_dir}/{dataset_name}.valid.inter", sep="\t", index=False)

Se realizan iteraciones para encontrar la mejor combinación de valores de hiperparámetros del modelo. Se hace variar la cantidad de épocas (epochs), embedding size, número de capas (n_layers), y learning rate utilizando listas, eligiendo valores a partir de los recomendados por la librería RecBole y los estándar en investigación. Para esto se modifica el config_dict del modelo en donde se definen los valores de parámetros. Para cada iteración, se registran las métricas obtenidos por el modelo usando cada combinación de hiperparámetros, y finalmente se muestra la que obtuvo mejor valor. Se utilizará la combinación que alcance mejor valor en las métricas de NDCG@10, Precision@10, y Recall@10:

In [ ]:
import os
import pandas as pd
import torch
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import LightGCN
from recbole.trainer import Trainer
from recbole.utils.case_study import full_sort_topk

#valores de parámetros a testear:
lista_epochs = [10, 20]
lista_embedding_size = [64, 128, 256]
lista_n_layers = [2, 3]
lista_learning_rate = [0.0005, 0.001, 0.002]

#registro de mejores valores de métricas y la combinación de hiperparámetros que las produjo
record_ndcg10 = 0.0
record_precision10 = 0.0
record_recall10 = 0.0

combinacion_record_ndcg10 = {}
combinacion_record_precision10 = {}
combinacion_record_recall10 = {}

#se ignora el warning "FutureWarning" para tener output más limpio:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

for epoch in lista_epochs:
    for emb_size in lista_embedding_size:
        for n_layer in lista_n_layers:
            for lr in lista_learning_rate:
                #diccionario de configuración requerido por RecBole para entrenar el modelo:
                config_dict_iteracion = {
                    "data_path": ".",
                    "model": "LightGCN",
                    "dataset": dataset_name,
                    "format": "custom",
                    "benchmark_filename" : ['train', 'valid', 'test'],
                    "dataset_file": {
                        "inter": {"train": f"{dataset_dir}/{dataset_name}.train.inter",
                            "valid": f"{dataset_dir}/{dataset_name}.valid.inter",
                            "test":  f"{dataset_dir}/{dataset_name}.test.inter"}
                    },
                    "field_separator": "\t",
                    "USER_ID_FIELD": "user_id",
                    "ITEM_ID_FIELD": "item_id",
                    "RATING_FIELD": "rating",
                    "load_col": {"inter": ["user_id", "item_id", "rating"]},
                    "field_type": {"user_id": "token","item_id": "token","rating": "float"},
                    "eval_args": {"mode": "full"},
                    "epochs": epoch,
                    "train_batch_size": 2048,
                    "eval_batch_size": 4096,
                    "embedding_size": emb_size,
                    "show_progress": True,
                    "learning_rate": lr,
                    "reg_weight": 1e-5,
                    "n_layers": n_layer,
                    "topk": [10],
                    "device": "cpu",
                    #umbral de rating para considerar un ítem como relevante.
                    #en este caso, todo ítem con rating >= 0.5 es considerado relevante.
                    "threshold": {"rating": 0.5} 
                }

                #se crea el objeto Config de RecBole que guarda toda la configuración
                # requerida por la librería, se carga el dataset, y se preparan los datos:
                config_iteracion = Config(config_dict=config_dict_iteracion)
                dataset = create_dataset(config_iteracion)
                train_data, valid_data, test_data = data_preparation(config_iteracion, dataset)

                #modelo LightGCN y el Trainer de RecBole:
                model_iteracion = LightGCN(config_iteracion, train_data.dataset).to(config_iteracion['device'])
                trainer_iteracion = Trainer(config_iteracion, model_iteracion)

                #entrenamiento
                trainer_iteracion.fit(train_data, valid_data)
                print()
                print("Entrenamiento finalizado iteración")

                test_result_iteracion = trainer_iteracion.evaluate(test_data)
                metricas_iteracion = dict(test_result_iteracion)

                if metricas_iteracion['ndcg@10'] > record_ndcg10:
                    record_ndcg10 = metricas_iteracion['ndcg@10']
                    combinacion_record_ndcg10 = {
                        "epochs": epoch,
                        "embedding_size": emb_size,
                        "n_layers": n_layer,
                        "learning_rate": lr
                    }

                if metricas_iteracion['precision@10'] > record_precision10:
                    record_precision10 = metricas_iteracion['precision@10']
                    combinacion_record_precision10 = {
                        "epochs": epoch,
                        "embedding_size": emb_size,
                        "n_layers": n_layer,
                        "learning_rate": lr
                    }
                
                if metricas_iteracion['recall@10'] > record_recall10:
                    record_recall10 = metricas_iteracion['recall@10']
                    combinacion_record_recall10 = {
                        "epochs": epoch,
                        "embedding_size": emb_size,
                        "n_layers": n_layer,
                        "learning_rate": lr
                    }

print("Mejores valores y combinaciones de hiperparámetros:")
print()
print("NDCG@10:", record_ndcg10)
print("Combinación de hiperparámetros:", combinacion_record_ndcg10)
print()
print("Precision@10:", record_precision10)
print("Combinación de hiperparámetros:", combinacion_record_precision10)
print()
print("Recall@10:", record_recall10)
print("Combinación de hiperparámetros:", combinacion_record_recall10)


Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado iteración

Entrenamiento finalizado it

Ahora, para evitar ejecutar el código de arriba cada vez (pues se demora apróx. 70 min) se registran aquí los valores de hiperparámetros que obtuvieron las mejores métricas, para usarlos en el  modelo final:

In [19]:
mejor_epoch = 20
mejor_embedding_size = 256
mejor_n_layers = 2
mejor_learning_rate = 0.002

Ahora, utilizando la mejor combinación de parámetros en base a los valores de métricas calculadas por la librería, se define el modelo final que se usará y se entrena:

In [20]:
import os
import pandas as pd
import torch
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import LightGCN
from recbole.trainer import Trainer
from recbole.utils.case_study import full_sort_topk

#diccionario de configuración requerido por RecBole para entrenar el modelo:
config_dict = {
    "data_path": ".",
    "model": "LightGCN",
    "dataset": dataset_name,
    "format": "custom",
    "benchmark_filename" : ['train', 'valid', 'test'],
    "dataset_file": {
        "inter": {
            "train": f"{dataset_dir}/{dataset_name}.train.inter",
            "valid": f"{dataset_dir}/{dataset_name}.valid.inter",
            "test":  f"{dataset_dir}/{dataset_name}.test.inter"
        }
    },
    "field_separator": "\t",
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "RATING_FIELD": "rating",
    "load_col": {"inter": ["user_id", "item_id", "rating"]},
    "field_type": {
        "user_id": "token",
        "item_id": "token",
        "rating": "float"
    },
    "eval_args": {"mode": "full"},
    "epochs": mejor_epoch,
    "train_batch_size": 2048,
    "eval_batch_size": 4096,
    "embedding_size": mejor_embedding_size,
    "show_progress": True,
    "learning_rate": mejor_learning_rate,
    "reg_weight": 1e-5,
    "n_layers": mejor_n_layers,
    "topk": [10],
    "device": "cpu",
    #umbral de rating para considerar un ítem como relevante.
    #en este caso, todo ítem con rating >= 0.5 es considerado relevante.
    "threshold": {"rating": 0.5} 
}

#se crea el objeto Config de RecBole que guarda toda la configuración
# requerida por la librería, se carga el dataset, y se preparan los datos:

config = Config(config_dict=config_dict)
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

#modelo LightGCN y el Trainer de RecBole:
model = LightGCN(config, train_data.dataset).to(config['device'])
trainer = Trainer(config, model)

#entrenamiento
trainer.fit(train_data, valid_data)
print()
print("Entrenamiento finalizado")


Entrenamiento finalizado


## 4- Obtención de scores de LightGCN

Ahora, a partir del entrenamiento anterior se obtienen las listas de recomendación (ranking) considerando todos los ítems del dataset para cada usuario. El objetivo de esto es obtener, para cada usuario, el score que LightGCN le asigna a cada ítem del dataset. Este score es usado por el modelo para armar los rankings, en donde un valor más alto significa un ítem más recomendado para cierto usuario. Se comienza usando las ids internas de la librería, para luego convertirlas a las originales del dataset:

In [21]:
#se guardan las ids internas de los usuarios presentes en el dataset,
# evitando la id = 0 que se asocia al ['PAD'] que usa internamente RecBole
dataset_obj = test_data.dataset
uid_series = [uid for uid in range(1, dataset_obj.user_num) 
              if test_data.uid2history_item[uid] is not None]

#se obtienen las listas de recomendación con todos los ítems para cada usuario
# se usa topk = número total de ítems (incluyendo el 'PAD' con id = 0)
topk = dataset.item_num
topk_result = full_sort_topk(uid_series, model, test_data, k=topk, device=config['device'])

Se guardan las recomendaciones y se convierten primero las ids internas de RecBole a las originales del dataset. Además, se guardan los scores obtenidos por LightGCN para cada par usuario-ítem con los que se hicieron las recomendaciones, en donde un mayor score significa que ese ítem es más recomendado para el usuario correspondiente.

In [22]:
#se convierten las ids de usuarios internas de RecBole a las originales del dataset
user_original_ids = dataset_obj.id2token('user_id', uid_series)

#se convierten las ids de ítems internas de RecBole a las originales del dataset ignorando el 'PAD'
topk_items_original = [
    [item for item in dataset_obj.id2token('item_id', topk_result.indices[i]) if item != "[PAD]"]
    for i in range(len(uid_series))
]

#se obtienen los scores asociados a cada ítem recomendado y se guardan en orden
# para cada usuario
scores_lightgcn = topk_result.values
topk_scores_list = [
    scores_lightgcn[i].tolist()
    for i in range(len(uid_series))
]

Se guardan las recomendaciones en dos diccionarios con las ids de usuario e ítem originales. El primer diccionario (top_lightgcn) tiene como llaves la id del usuario y su valor es una lista ordenada con las id de ítems recomendados:. El segundo diccionario (top_lightgcn_with_scores) tiene como llaves la id del usuario y su valor es otro diccionario con la forma "{item id : score LightGCN}", y de esta forma saber el orden de los ítems recomendados a cada usuario y cuál fue el score que LightGCN calculó para cada uno. Este score se usará para incluir la multimodalidad de datos de texto.

In [24]:
#diccionario final de recomendaciones simples (id de usuario → [ids de ítems]):
top_lightgcn = {int(user_id): [int(item) for item in items] for user_id, items in zip(user_original_ids, topk_items_original)}

#diccionario final de recomendaciones con scores (id de usuario → [(id ítem, score), ...]):
top_lightgcn_with_scores = {
    int(user_id): {int(item_id) : float(score) for item_id, score in zip(topk_items_original[i], topk_scores_list[i])}
    for i, user_id in enumerate(user_original_ids)
}

print("Ejemplo de 5 usuarios y sus primeras 5 recomendaciones:")
for user_id, items in list(top_lightgcn.items())[:5]:
    print(f"Usuario {user_id} → {items[:5]}")

print()

print("Ejemplo de 5 usuarios y sus primeras 5 recomendaciones con scores:")
for user_id, dict_item_score in list(top_lightgcn_with_scores.items())[:5]:
    print(f"Usuario {user_id} → {list(dict_item_score.items())[:5]}")

Ejemplo de 5 usuarios y sus primeras 5 recomendaciones:
Usuario 731 → [440, 304930, 4000, 236390, 252490]
Usuario 3128 → [275850, 427520, 244850, 294100, 233860]
Usuario 4232 → [550, 238320, 12210, 440, 220]
Usuario 8297 → [1091500, 629730, 945360, 1172620, 431960]
Usuario 9905 → [304930, 444090, 4000, 550, 105600]

Ejemplo de 5 usuarios y sus primeras 5 recomendaciones con scores:
Usuario 731 → [(440, 6.540383338928223), (304930, 5.660223960876465), (4000, 5.572798728942871), (236390, 5.359306335449219), (252490, 5.086994171142578)]
Usuario 3128 → [(275850, 8.7120361328125), (427520, 7.248980522155762), (244850, 6.878946781158447), (294100, 6.67802619934082), (233860, 6.504183769226074)]
Usuario 4232 → [(550, 6.4183149337768555), (238320, 6.064563751220703), (12210, 5.757477760314941), (440, 5.707176685333252), (220, 5.705965995788574)]
Usuario 8297 → [(1091500, 5.052934646606445), (629730, 4.9911956787109375), (945360, 4.970223426818848), (1172620, 4.937317371368408), (431960, 4.8508

Usuarios con recomendaciones (en total el dataset tiene 3000):

In [25]:
len(top_lightgcn)

9906

## 5- Cálculo de scores combinados (multimodales)

Ahora se calculan los scores combinados (o multimodales) como la suma ponderada de los scores de lightgcn (sección 4) con los scores de la similitud de embeddings (sección 2) de la siguiente forma: score_combinado = alfa * score_lightgcn + (1-alfa) * score_embeddings_texto. Se comienza con alfa = 0.5.

Esto se hace para cada par usuario_ítem posible y se guardan en un diccionario con la forma: {id_usuario : {id_item : score_combinado_item}}.

In [26]:
alfa = 0.5

scores_combinados = {}

for user_id in top_lightgcn_with_scores.keys():

    scores_combinados[user_id] = {}
    items = scores_user_item_embeddings[user_id].keys()

    for item_id in items:
        score_lightgcn = top_lightgcn_with_scores[user_id].get(item_id, 0.0)
        score_embedding = scores_user_item_embeddings[user_id].get(item_id, 0.0)

        score_combinado = alfa * score_lightgcn + (1 - alfa) * score_embedding

        scores_combinados[user_id][item_id] = score_combinado


## 6- Generación de nuevas recomendaciones con scores combinados (multimodales)

Ya con los scores combinados calculados, se debe armar la lista de recomendación top 10. para cada usuario. Para ello, se toman las id de los 10 ítems con mayor score combinado para el usuario. Se guardan en el diccionario "top10_lightgcn_texto" con la forma "{id usuario : lista ordenada de las 10 id de ítems recomendados}"

In [27]:
top10_lightgcn_texto = {}

for user_id in scores_combinados.keys():

    item_score_dict = scores_combinados[user_id]
    items_ordenados = sorted(item_score_dict.items(), key=lambda x: x[1], reverse=True)

    top10_items = [item_id for item_id, score in items_ordenados[:10]]

    top10_lightgcn_texto[user_id] = top10_items

Ejemplo para los primeros 5 usuarios:

In [28]:
print("Ejemplo de 5 usuarios y sus recomendaciones (top 10):")
for user_id, items in list(top10_lightgcn_texto.items())[:5]:
    print(f"Usuario {user_id} → {items}")

Ejemplo de 5 usuarios y sus recomendaciones (top 10):
Usuario 731 → [440, 4000, 236390, 252490, 431960, 304930, 346110, 242760, 220200, 218620]
Usuario 3128 → [275850, 427520, 244850, 294100, 233860, 392160, 261550, 289070, 359320, 526870]
Usuario 4232 → [550, 238320, 12210, 440, 220, 239140, 271590, 70, 218620, 379720]
Usuario 8297 → [1091500, 629730, 945360, 1172620, 431960, 823500, 284160, 438100, 620980, 739630]
Usuario 9905 → [444090, 304930, 4000, 550, 105600, 620, 945360, 218620, 204360, 49520]


Ejemplo recomendación usuario con ID = 731:

In [ ]:
print("Ejemplo de usuario ID = 781 y sus recomendaciones:")
print("IDs videojuegos recomendados:")
print(top10_lightgcn_texto[731])
print("Títulos videojuegos recomendados:")
print([df_games[df_games["app_id"] == item]["title"].values[0] for item in top10_lightgcn_texto[731]])

## 7- Métricas

Para facilitar el cálculo, se crea el diccionario dict_items_relevantes que tiene como llave la id de usuario (para cada uno de los que tienen una recomendación top 10) y el valor es el set de ítems relevantes del dataset de test:

In [37]:
dic_items_relevantes = {}

for user_id in top10_lightgcn_texto.keys():
    relevantes_usuario = set(df_test[(df_test["user_id"] == user_id) & (df_test["rating"] >= 0.5)]["app_id"])
    dic_items_relevantes[user_id] = relevantes_usuario

Cálculo NDCG@10:

In [ ]:
import numpy as np

def ndcg_at_k(items_recomendados, items_relevantes, k=10):

    dcg = 0.0
    for i, item in enumerate(items_recomendados[:k]):
        if item in items_relevantes:
            dcg += 1 / np.log2(i + 2)

    ideal_hits = min(len(items_relevantes), k)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_hits))

    if idcg == 0:
        return 0.0

    return dcg / idcg

ndcg_scores = {}
for user_id, items in top10_lightgcn_texto.items():

    items_relevantes_usuario = dic_items_relevantes[user_id]

    ndcg_scores[user_id] = ndcg_at_k(items, items_relevantes_usuario, k=10)

ndcg_lightgcn = np.mean(list(ndcg_scores.values()))
print(f"NDCG@10: {ndcg_lightgcn:.4f}")

NDCG@10: 0.02839752593857097


Cálculo de Precision@10:

In [ ]:
def precision_at_k(topk_dict, relevantes_dict, k=10):
    precisions = []

    for user_id, relevants in relevantes_dict.items():

        if user_id not in topk_dict:
            continue

        topk_items = topk_dict[user_id][:k]

        interseccion = len(set(topk_items) & relevants)

        precision_u = interseccion / k
        precisions.append(precision_u)

    return sum(precisions) / len(precisions) if precisions else 0.0

precision_lightgcn = precision_at_k(top10_lightgcn_texto, dic_items_relevantes, k=10)
print(f"Precision@10: {precision_lightgcn:.4f}")


Precision@10: 0.0065


Cálculo de Recall@10:

In [ ]:
def recall_at_k(topk_dict, relevantes_dict, k=10):
    recalls = []

    for user_id, relevants in relevantes_dict.items():

        if user_id not in topk_dict or len(relevants) == 0:
            continue 

        topk_items = topk_dict[user_id][:k]

        interseccion = len(set(topk_items) & relevants)

        recall_u = interseccion / len(relevants)
        recalls.append(recall_u)

    return sum(recalls) / len(recalls) if recalls else 0.0

recall_lightgcn = recall_at_k(top10_lightgcn_texto, dic_items_relevantes, k=10)
print(f"Recall@10: {recall_lightgcn:.4f}")


Recall@10: 0.0631


Cálculo de HitRate@10:

In [ ]:
def hitrate_at_k(topk_dict, relevantes_dict, k=10):
    hitrates = []

    for user_id, relevants in relevantes_dict.items():

        if user_id not in topk_dict:
            continue

        topk_items = topk_dict[user_id][:k]

        interseccion = set(topk_items) & set(relevants)

        hitrate_u = 1 if len(interseccion) > 0 else 0
        
        hitrates.append(hitrate_u)

    return sum(hitrates) / len(hitrates) if hitrates else 0.0
    
hitrate_lightgcn = hitrate_at_k(top10_lightgcn_texto, dic_items_relevantes, k=10)
print(f"HitRate@10: {hitrate_lightgcn:.4f}")

HitScore@10: 0.0654


Otra métrica que se utilizará es el F1 Score. Se calcula y guarda en una variable:

In [44]:
f1score_lightgcn = 2 * (precision_lightgcn * recall_lightgcn) / (precision_lightgcn + recall_lightgcn)
print(f"F1 Score@10: {f1score_lightgcn:.4f}")

F1 Score@10: 0.0119


Ahora, se calcula el MAP@K. Se comienzan por definir dos funciones base, una para calcular el Average Precision at K (AP@K) para un usuario y luego otra para calcular el MAP@K como el promedio del AP@K de todos los usuarios. Finalmente, se calcula el MAP@10:

In [53]:
def ap_at_k(items_relevantes, items_recomendados, k):
    if len(items_recomendados) > k:
        items_recomendados = items_recomendados[:k]
        
    score = 0
    num_hits = 0
    for i, p in enumerate(items_recomendados):
        if p in items_relevantes and p not in items_recomendados[:i]:
            num_hits += 1
            score += (num_hits / (i + 1))
    
    if not items_relevantes:
        return 0

    return score / min(len(items_relevantes), k)

def map_at_k(relevantes_dict, dict_recomendados, k):
    ap_scores = []
    for user_id, lista_recomendados in dict_recomendados.items():
        items_relevantes = relevantes_dict[user_id]
        ap = ap_at_k(items_relevantes, lista_recomendados, k)
        ap_scores.append(ap)
    
    return sum(ap_scores) / len(ap_scores)

map_lightgcn = map_at_k(dic_items_relevantes, top10_lightgcn_texto, k=10)
print(f"MAP@10: {map_lightgcn:.4f}")

MAP@10: 0.0191


Ahora, se calcula la diversidad promedio de las recomendaciones (Diversity). Para esto se analizan los géneros de videojuegos recomendados y la métrica representa cuántos géneros distintos de videojuegos se recomiendan en promedio. Se comienza definiendo dos funciones para realizar los cálculos y luego se obtiene Diversity: 

In [50]:
#código basado en el elaborado por Nicolás Bueno y Felipe Fuentes en Tarea del curso.

#función para calcular la cantidad de géneros distintos de videojuegos en la lista de recomendación de un usuario
def diversity_user(items_recomendados, dict_item_genero):
    if not items_recomendados:
        return 0
    
    unique_generos = set()
    for item_id in items_recomendados:
        if item_id in dict_item_genero:
            #se recorre la lista de tags (géneros) asociados al videojuego
            for tag in dict_item_genero[item_id]:
                unique_generos.add(tag)

    return float(len(unique_generos))

#función para calcular la cantidad promedio de géneros distintos de videojuegos en todas las listas de recomendación generadas
def diversity(recomendaciones, dict_item_genero):
    total = 0
    cant_usuarios = 0
    for recs in recomendaciones.values():
        total += diversity_user(recs, dict_item_genero)
        cant_usuarios += 1
        
    return total / max(cant_usuarios, 1)

#los géneros de un videojuego se consideran como los "tags" asociados en el archivo de metadata (guardado en en el dataframe games_metadata)
dict_item_genero = dict(zip(games_metadata["app_id"].astype(int), games_metadata["tags"]))
diversity_lightgcn = diversity(top10_lightgcn_texto, dict_item_genero)
print(f"Diversidad promedio: {diversity_lightgcn}")

Diversidad promedio: 18.660811629315567


En resumen, las métricas calculadas del modelo son:

In [ ]:
print("Resumen de las métricas:")
print()
print(f"Recall@10: {recall_lightgcn:.4f}")
print(f"Precision@10: {precision_lightgcn:.4f}")
print(f"F1 Score@10: {f1score_lightgcn:.4f}")
print(f"NDCG@10: {ndcg_lightgcn:.4f}")
print(f"HitRate@10: {hitrate_lightgcn:.4f}")
print(f"MAP@10: {map_lightgcn:.4f}")
print(f"Diversity: {diversity_lightgcn:.4f}")

Resumen de las métricas:

Recall@10: 0.0631
Precision@10: 0.0065
F1 Score@10: 0.0119
NDCG@10: 0.0284
HitScore@10: 0.0654
MAP@10: 0.0191
Diversity: 18.6608


## 8- Referencias

[1] Deng, K., He, X., Li, Y., Wang, M., Wang, X., & Zhang, Y. (2020). LightGCN: Simplifying and powering graph Convolution Network for recommendation. ArXiv. Obtenido de http://arxiv.org/abs/2002.02126

[2] Diapositivas de la clase "Clase de Evaluación: metricas de error y ranking", como apoyo para implementar cálculo de métricas. Enlace: https://github.com/PUC-RecSys-Class/RecSysPUC-2025-2/blob/master/clases/s3_c1-metricas_v3.pdf

[3] Documentación de RecBole 1.2.1: https://recbole.io/docs/

[4] Documentación modelo LightGCN de Recbole: https://recbole.io/docs/user_guide/model/general/lightgcn.html

[5] Repositorio Recbole en GitHub: https://github.com/RUCAIBox/RecBole

[6] Práctico de métricas del curso del semestre pasado, utilizado como base principalmente para programar las métrica de Diversity. Enlace: https://github.com/PUC-RecSys-Class/RecSysPUC-2025-2/blob/master/practicos/pr%C3%A1ctico_m%C3%A9tricas.ipynb

[7] Dataset principal: "Game Recommendations on Steam: A dataset of games, users and reviews for building recommendation systems". Anton Kozyriev, 2024. Kaggle. Link: https://www.kaggle.com/datasets/antonkozyriev/game-recommendations-on-steam?select=recommendations.csv

[8] Metadata adicional de videojuegos de Steam: "Steam Store Games (Clean dataset): Combined data of 27,000 games scraped from Steam and SteamSpy APIs". Nik Davis, 2019

[9] Uso de IA. Se consultó a ChatGPT sobre la librería RecBole, el uso de funciones y formatos de LightGCN, y errores. Link al chat: https://chatgpt.com/share/68ffb7df-f778-8008-b287-14268d81d203